# Skill Barter Recommendation Model

This notebook builds a **hybrid recommendation system** on top of the uploaded skill barter dataset.

## What this notebook covers
1. Load and inspect the dataset
2. Clean and preprocess the data
3. Build a **skill recommendation model** to predict which skill a user may request
4. Evaluate the model with **Top-1 / Top-3 / Top-5** accuracy
5. Build a **partner recommendation engine** for barter matching between users
6. Generate example recommendations

In [15]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
from IPython.display import display

## 1. Load the dataset

The notebook assumes `skill_barter_dataset.csv` is placed in the same folder as this notebook.

In [16]:
DATA_PATH = Path("skill_barter_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.name}. Place the CSV file in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 7)


,user_id,skill_offered,skill_requested,experience_level,category,rating,interaction
0,U595,Figma,Node.js,Beginner,Design,4,view
1,U140,Python,Spring Boot,Intermediate,Programming,3,request
2,U138,Spring Boot,Node.js,Advanced,Programming,3,request
3,U736,Node.js,Spring Boot,Advanced,Programming,4,request
4,U602,Spring Boot,React,Beginner,Programming,4,view


## 2. Quick dataset overview

In [17]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "missing_values": [int(df[col].isna().sum()) for col in df.columns],
    "unique_values": [int(df[col].nunique()) for col in df.columns],
})

summary

,column,dtype,missing_values,unique_values
0,user_id,object,0,1000
1,skill_offered,object,0,15
2,skill_requested,object,0,15
3,experience_level,object,0,3
4,category,object,0,5
5,rating,int64,0,3
6,interaction,object,0,3


In [18]:
print("Number of users:", df["user_id"].nunique())
print("Unique offered skills:", df["skill_offered"].nunique())
print("Unique requested skills:", df["skill_requested"].nunique())

print("\nTop offered skills:")
print(df["skill_offered"].value_counts().head(10))

print("\nTop requested skills:")
print(df["skill_requested"].value_counts().head(10))

print("\nInteraction distribution:")
print(df["interaction"].value_counts())

Number of users: 1000
Unique offered skills: 15
Unique requested skills: 15

Top offered skills:
skill_offered
React               693
Power BI            680
Java                680
Machine Learning    675
Data Analysis       671
Python              671
UI/UX Design        670
Deep Learning       669
JavaScript          667
SQL                 666
Name: count, dtype: int64

Top requested skills:
skill_requested
UI/UX Design        709
Machine Learning    709
Power BI            698
Figma               696
Node.js             695
Java                681
Excel               671
JavaScript          663
Deep Learning       655
Python              651
Name: count, dtype: int64

Interaction distribution:
interaction
completed    5016
request      2962
view         2022
Name: count, dtype: int64


## 3. Data cleaning and preprocessing

In [19]:
clean_df = df.copy()

text_cols = [
    "user_id",
    "skill_offered",
    "skill_requested",
    "experience_level",
    "category",
    "interaction",
]

for col in text_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip()

clean_df["rating"] = pd.to_numeric(clean_df["rating"], errors="coerce")
clean_df["rating"] = clean_df["rating"].fillna(clean_df["rating"].median())

clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Cleaned dataset shape:", clean_df.shape)
clean_df.head()

Cleaned dataset shape: (9986, 7)


,user_id,skill_offered,skill_requested,experience_level,category,rating,interaction
0,U595,Figma,Node.js,Beginner,Design,4,view
1,U140,Python,Spring Boot,Intermediate,Programming,3,request
2,U138,Spring Boot,Node.js,Advanced,Programming,3,request
3,U736,Node.js,Spring Boot,Advanced,Programming,4,request
4,U602,Spring Boot,React,Beginner,Programming,4,view


## 4. Build a skill recommendation model

### Goal
Predict the most likely **`skill_requested`** using:
- `skill_offered`
- `experience_level`
- `category`
- `rating`
- `interaction`

This gives a practical **Top-N recommendation model** for the barter platform.

In [20]:
feature_cols = ["skill_offered", "experience_level", "category", "rating", "interaction"]
target_col = "skill_requested"

X = clean_df[feature_cols]
y = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = ["skill_offered", "experience_level", "category", "interaction"]
numeric_features = ["rating"]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer([
    ("cat", categorical_pipeline, categorical_features),
    ("num", numeric_pipeline, numeric_features),
])

skill_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000)),
])

skill_model.fit(X_train, y_train)
print("Skill recommendation model trained successfully.")

Skill recommendation model trained successfully.


## 5. Evaluate the skill recommendation model

In [21]:
y_pred = skill_model.predict(X_test)
y_proba = skill_model.predict_proba(X_test)

top1_accuracy = accuracy_score(y_test, y_pred)
top3_accuracy = top_k_accuracy_score(y_test, y_proba, k=3, labels=skill_model.classes_)
top5_accuracy = top_k_accuracy_score(y_test, y_proba, k=5, labels=skill_model.classes_)

metrics_df = pd.DataFrame({
    "metric": ["Top-1 Accuracy", "Top-3 Accuracy", "Top-5 Accuracy"],
    "value": [top1_accuracy, top3_accuracy, top5_accuracy],
})

metrics_df["value"] = metrics_df["value"].round(4)
metrics_df

,metric,value
0,Top-1 Accuracy,0.0731
1,Top-3 Accuracy,0.2157
2,Top-5 Accuracy,0.3554


## 6. Helper function: recommend requested skills

This function returns the **Top-N skill recommendations** for a given input profile.

In [22]:
def recommend_requested_skills(
    skill_offered,
    experience_level,
    category,
    rating,
    interaction="request",
    top_n=5,
    exclude_same_skill=True,
):
    sample = pd.DataFrame([{
        "skill_offered": skill_offered,
        "experience_level": experience_level,
        "category": category,
        "rating": rating,
        "interaction": interaction,
    }])

    probabilities = skill_model.predict_proba(sample)[0]
    classes = skill_model.classes_

    recs = pd.DataFrame({
        "recommended_skill": classes,
        "score": probabilities,
    }).sort_values("score", ascending=False)

    if exclude_same_skill:
        recs = recs[recs["recommended_skill"] != skill_offered]

    recs["score"] = recs["score"].round(4)
    return recs.head(top_n).reset_index(drop=True)

In [23]:
example_row = clean_df.iloc[0]

print("Example input profile:")
display(example_row.to_frame().T)

print("\nTop recommended requested skills:")
recommend_requested_skills(
    skill_offered=example_row["skill_offered"],
    experience_level=example_row["experience_level"],
    category=example_row["category"],
    rating=example_row["rating"],
    interaction=example_row["interaction"],
    top_n=5,
)

Example input profile:


,user_id,skill_offered,skill_requested,experience_level,category,rating,interaction
0,U595,Figma,Node.js,Beginner,Design,4,view



Top recommended requested skills:


,recommended_skill,score
0,Node.js,0.1034
1,Power BI,0.1002
2,Python,0.0810
3,Spring Boot,0.0794
4,Java,0.0732


## 7. Build user-level profiles for partner recommendation

Now we create a **barter partner recommendation engine**.

### Logic
A strong barter partner is someone who:
- offers a skill that the target user wants
- wants a skill that the target user offers
- has a strong content/profile similarity
- has good rating and completion history

In [24]:
def mode_or_first(series):
    mode_values = series.mode()
    if len(mode_values) > 0:
        return mode_values.iloc[0]
    return series.iloc[0]

user_profiles = (
    clean_df.groupby("user_id")
    .agg(
        offered_skills=("skill_offered", lambda x: sorted(set(x))),
        requested_skills=("skill_requested", lambda x: sorted(set(x))),
        categories=("category", lambda x: sorted(set(x))),
        experience_level=("experience_level", mode_or_first),
        avg_rating=("rating", "mean"),
        completed_count=("interaction", lambda x: int((x == "completed").sum())),
        total_records=("interaction", "size"),
    )
    .reset_index()
)

user_profiles["profile_text"] = user_profiles.apply(
    lambda row: " ".join(
        [f"offer_{skill.replace(' ', '_')}" for skill in row["offered_skills"]]
        + [f"want_{skill.replace(' ', '_')}" for skill in row["requested_skills"]]
        + [f"cat_{cat.replace(' ', '_')}" for cat in row["categories"]]
        + [f"exp_{row['experience_level']}"]
    ),
    axis=1,
)

print("User profile shape:", user_profiles.shape)
user_profiles.head()

User profile shape: (1000, 9)


,user_id,offered_skills,requested_skills,categories,experience_level,avg_rating,completed_count,total_records,profile_text
0,U1,"[Data Analysis, Java, Node.js, Power BI, Pytho...","[Excel, Figma, JavaScript, Node.js, Spring Boo...","[Analytics, Design, Programming]",Beginner,4.142857,3,7,offer_Data_Analysis offer_Java offer_Node.js o...
1,U10,"[Data Analysis, Figma, Java, JavaScript, Machi...","[Deep Learning, Figma, Java, JavaScript, Machi...","[AI, Analytics, Database, Design, Programming]",Beginner,4.166667,4,12,offer_Data_Analysis offer_Figma offer_Java off...
2,U100,"[Data Analysis, Data Visualization, JavaScript...","[Deep Learning, JavaScript, Node.js, Power BI,...","[Analytics, Database, Design, Programming]",Beginner,4.000000,4,10,offer_Data_Analysis offer_Data_Visualization o...
3,U1000,"[Data Analysis, Data Visualization, JavaScript...","[Deep Learning, Figma, Java, Node.js, Power BI...","[Analytics, Database, Design, Programming]",Advanced,3.888889,4,9,offer_Data_Analysis offer_Data_Visualization o...
4,U101,"[Data Analysis, Data Visualization, Deep Learn...","[Data Analysis, Deep Learning, Excel, Java, Ja...","[AI, Analytics, Database, Design, Programming]",Advanced,4.000000,8,13,offer_Data_Analysis offer_Data_Visualization o...


In [25]:
tfidf = TfidfVectorizer()
profile_matrix = tfidf.fit_transform(user_profiles["profile_text"])
similarity_matrix = cosine_similarity(profile_matrix)

user_to_index = {user_id: idx for idx, user_id in enumerate(user_profiles["user_id"])}
profile_lookup = user_profiles.set_index("user_id").to_dict(orient="index")

experience_rank = {"Beginner": 0, "Intermediate": 1, "Advanced": 2}

## 8. Helper function: recommend barter partners

In [26]:
def recommend_partners(user_id, top_n=5):
    if user_id not in profile_lookup:
        raise ValueError(f"User ID {user_id} not found in the dataset.")

    target = profile_lookup[user_id]
    target_offered = set(target["offered_skills"])
    target_requested = set(target["requested_skills"])
    target_index = user_to_index[user_id]

    recommendations = []

    for candidate_id, candidate in profile_lookup.items():
        if candidate_id == user_id:
            continue

        candidate_offered = set(candidate["offered_skills"])
        candidate_requested = set(candidate["requested_skills"])

        forward_match = target_requested.intersection(candidate_offered)
        reverse_match = target_offered.intersection(candidate_requested)

        if not forward_match and not reverse_match:
            continue

        forward_score = len(forward_match) / max(len(target_requested), 1)
        reverse_score = len(reverse_match) / max(len(target_offered), 1)
        barter_score = 0.5 * forward_score + 0.5 * reverse_score

        content_score = similarity_matrix[target_index, user_to_index[candidate_id]]
        rating_score = candidate["avg_rating"] / 5.0
        completion_score = candidate["completed_count"] / max(candidate["total_records"], 1)

        exp_gap = abs(
            experience_rank.get(target["experience_level"], 1)
            - experience_rank.get(candidate["experience_level"], 1)
        )
        experience_score = 1 - (exp_gap / 2)

        final_score = (
            0.50 * barter_score
            + 0.20 * content_score
            + 0.15 * rating_score
            + 0.10 * completion_score
            + 0.05 * experience_score
        )

        recommendations.append({
            "candidate_user_id": candidate_id,
            "match_score": round(float(final_score), 4),
            "skills_they_offer_you_need": sorted(forward_match),
            "skills_they_want_from_you": sorted(reverse_match),
            "candidate_experience": candidate["experience_level"],
            "candidate_avg_rating": round(float(candidate["avg_rating"]), 2),
            "completion_rate": round(float(completion_score), 4),
            "content_similarity": round(float(content_score), 4),
        })

    results = pd.DataFrame(recommendations)

    if results.empty:
        return results

    return (
        results
        .sort_values(
            by=["match_score", "content_similarity", "candidate_avg_rating"],
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

In [27]:
example_user_id = user_profiles["user_id"].iloc[0]
print("Example user:", example_user_id)
recommend_partners(example_user_id, top_n=5)

Example user: U1


,candidate_user_id,match_score,skills_they_offer_you_need,skills_they_want_from_you,candidate_experience,candidate_avg_rating,completion_rate,content_similarity
0,U12,0.8367,"[Figma, JavaScript, Node.js, Spring Boot, UI/U...","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,4.00,0.6875,0.6983
1,U665,0.8163,"[Excel, Figma, JavaScript, Node.js, Spring Boot]","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,4.13,0.6000,0.6199
2,U514,0.8106,"[Excel, Figma, JavaScript, Node.js, UI/UX Design]","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,3.69,0.5385,0.6884
3,U155,0.7947,"[Excel, Figma, JavaScript, Spring Boot, UI/UX ...","[Data Analysis, Node.js, Power BI, Python, UI/...",Beginner,4.64,0.6364,0.6266
4,U567,0.7850,"[Excel, Figma, JavaScript, Node.js, Spring Boo...","[Data Analysis, Power BI, Python, UI/UX Design]",Beginner,4.00,0.6250,0.6790


## 9. Optional: inspect one user profile before recommending

In [28]:
target_user_id = example_user_id

print("Target user profile:")
display(user_profiles[user_profiles["user_id"] == target_user_id])

print("\nRecommended barter partners:")
display(recommend_partners(target_user_id, top_n=10))

Target user profile:


,user_id,offered_skills,requested_skills,categories,experience_level,avg_rating,completed_count,total_records,profile_text
0,U1,"[Data Analysis, Java, Node.js, Power BI, Pytho...","[Excel, Figma, JavaScript, Node.js, Spring Boo...","[Analytics, Design, Programming]",Beginner,4.142857,3,7,offer_Data_Analysis offer_Java offer_Node.js o...



Recommended barter partners:


,candidate_user_id,match_score,skills_they_offer_you_need,skills_they_want_from_you,candidate_experience,candidate_avg_rating,completion_rate,content_similarity
0,U12,0.8367,"[Figma, JavaScript, Node.js, Spring Boot, UI/U...","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,4.00,0.6875,0.6983
1,U665,0.8163,"[Excel, Figma, JavaScript, Node.js, Spring Boot]","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,4.13,0.6000,0.6199
2,U514,0.8106,"[Excel, Figma, JavaScript, Node.js, UI/UX Design]","[Data Analysis, Java, Node.js, Power BI, Pytho...",Beginner,3.69,0.5385,0.6884
3,U155,0.7947,"[Excel, Figma, JavaScript, Spring Boot, UI/UX ...","[Data Analysis, Node.js, Power BI, Python, UI/...",Beginner,4.64,0.6364,0.6266
4,U567,0.7850,"[Excel, Figma, JavaScript, Node.js, Spring Boo...","[Data Analysis, Power BI, Python, UI/UX Design]",Beginner,4.00,0.6250,0.6790
5,U138,0.7827,"[Excel, Figma, Node.js, Spring Boot, UI/UX Des...","[Java, Node.js, Power BI, Python, UI/UX Design]",Beginner,3.79,0.5000,0.7625
6,U508,0.7665,"[Excel, JavaScript, Node.js, Spring Boot, UI/U...","[Data Analysis, Java, Node.js, Power BI, Pytho...",Advanced,4.29,0.4286,0.6838
7,U762,0.7663,"[Excel, Figma, JavaScript, Node.js, Spring Boo...","[Data Analysis, Java, Power BI, Python, UI/UX ...",Intermediate,4.43,0.5000,0.5005
8,U79,0.7524,"[Excel, Figma, JavaScript, Spring Boot, UI/UX ...","[Data Analysis, Java, Power BI, Python, UI/UX ...",Beginner,4.00,0.6364,0.5106
9,U600,0.7449,"[Excel, Figma, JavaScript, Spring Boot, UI/UX ...","[Data Analysis, Node.js, Python, UI/UX Design]",Beginner,3.83,0.6667,0.6910
